# Calibration: how big is a price gap?

Every price in the game is measured in one unit, `g`. This notebook sets `g` from real market data so the numbers the AI client sees are realistic for an off-the-run 10-year US Treasury note.

1. **C1. Daily yield volatility** (`sigma_bp`, basis points per day) from the Federal Reserve's Gürkaynak–Sack–Wright yield curve, which is fitted to off-the-run issues.
2. **C2. Cross-checks** against the on-the-run 10-year yield (FRED `DGS10`) and the MOVE index of implied Treasury volatility.
3. **C3. Convert to price** with the modified duration of a par 9.75-year note (the first off-the-run) at the window's average yield, giving `sigma` in price points per day. Then `g = 0.1 × sigma`.

Result of record (window 2025-09-02 to 2026-09-11, 258 days): `sigma_bp` 4.31bp/day, `g` = 0.0339 points, or about 1/32nd, $3,387 on $10mm, 0.43bp of yield. That makes the median dealer margin 0.43bp, against a real-world range of roughly 0.25–1bp for this trade.

The values were saved to `config.yaml` on 2026-09-22. The last cell now **checks** this calculation against `config.yaml` instead of rewriting it. It needs internet access, and the Fed may revise its curve, so a rerun can differ in the last decimals.

In [ ]:
import io, os
from pathlib import Path

import pandas as pd
import requests
import yfinance as yf

# Run from the repository root, wherever the notebook was opened from
ROOT = Path.cwd() if (Path.cwd() / "run.py").exists() else Path.cwd().parent
os.chdir(ROOT)

WINDOW = ("2025-09-01", "2026-09-11")   # the window of record

C1:  daily yield vol           sigma_bp     (basis points per day)
C2:  sanity-check it against two other sources
C3:  × modified duration  →    sigma        (price points per day)
     × LAMBDA (0.1)      →     g            (the gap scale)

In [ ]:
URL = "https://www.federalreserve.gov/data/yield-curve-tables/feds200628.csv"
raw = requests.get(URL, timeout=60).text.splitlines()

print(len(raw), "lines")
print("\n".join(raw[:10])[:600])

start = next(i for i, l in enumerate(raw) if l.startswith("Date"))
print("header at line", start)

gsw = pd.read_csv(io.StringIO("\n".join(raw[start:])),
                   index_col=0, parse_dates=True)

print(gsw.shape)
print([c for c in gsw.columns if c.startswith("SVENY")][:12])


17068 lines
"Note: This is not an official Federal Reserve statistical release. Because this is a staff research product and not an official statistical release, it is subject to delay, revision, or methodological changes without advance notice."

Series,Compounding Convention,Mnemonic(s)
Zero-coupon yield,Continuously Compounded,SVENYXX
Par yield,Coupon-Equivalent,SVENPYXX
Instantaneous forward rate,Continuously Compounded,SVENFXX
One-year forward rate,Coupon-Equivalent,SVEN1FXX
Parameters,N/A,BETA0 to TAU2

Date,BETA0,BETA1,BETA2,BETA3,SVEN1F01,SVEN1F04,SVEN1F09,SVENF01,SVENF02,SVENF03,SVENF04,SVENF05,S
header at line 9


(17058, 99)
['SVENY01', 'SVENY02', 'SVENY03', 'SVENY04', 'SVENY05', 'SVENY06', 'SVENY07', 'SVENY08', 'SVENY09', 'SVENY10', 'SVENY11', 'SVENY12']


In [ ]:
#Extract 10 year series and window it 

y10 = pd.to_numeric(gsw["SVENY10"], errors="coerce").dropna()
print("full range:", y10.index.min().date(), "→", y10.index.max().date())
print(y10.tail())

y10w = y10.loc[WINDOW[0]:WINDOW[1]]
print(len(y10w), "observations in window")
print("y10 window range:", y10w.index.min().date(), "→", y10w.index.max().date())


full range: 1971-08-16 → 2026-10-30
Date
2026-10-26    4.9848
2026-10-27    4.9743
2026-10-28    5.0377
2026-10-29    4.9727
2026-10-30    4.9324
Name: SVENY10, dtype: float64
258 observations in window
y10 window range: 2025-09-02 → 2026-09-11


In [ ]:
#Calculate vol for data calibration

sigma_bp = (y10w * 100).diff().dropna().std()
print(f"sigma_bp = {sigma_bp:.3f} bp/day")

sigma_bp = 4.307 bp/day


In [ ]:
# Cross-checking the data (off-run from FRED vs on-run from YFinance)

dgs10 = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10",
                    index_col=0, parse_dates=True, na_values=".").iloc[:, 0].dropna()

sigma_bp_onrun = (dgs10.loc[WINDOW[0]:WINDOW[1]] * 100).diff().dropna().std()
print(f"off-the-run (GSW):  {sigma_bp:.3f} bp/day")
print(f"on-the-run (DGS10): {sigma_bp_onrun:.3f} bp/day")

raw_move = yf.download("^MOVE", start=WINDOW[0], end="2026-09-12", progress=False, auto_adjust=True)
close = raw_move["Close"]
if hasattr(close, "columns"):        # newer yfinance returns MultiIndex columns
    close = close.iloc[:, 0]

sigma_bp_implied = float(close.mean()) / 252 ** 0.5
print(f"MOVE mean (annualised): {float(close.mean()):.1f} bp")
print(f"implied daily:          {sigma_bp_implied:.3f} bp/day")

off-the-run (GSW):  4.307 bp/day
on-the-run (DGS10): 4.094 bp/day


MOVE mean (annualised): 73.1 bp
implied daily:          4.603 bp/day


In [ ]:
# Check correlation between the SVENY10 and DGS10 series 

common = y10w.index.intersection(dgs10.index)
a = (y10.loc[common] * 100).diff().dropna()
b = (dgs10.loc[common] * 100).diff().dropna()

print(f"{len(common)} common dates")
print(f"GSW   sd {a.std():.3f} bp")
print(f"DGS10 sd {b.std():.3f} bp")
print(f"correlation of daily changes: {a.corr(b):.3f}")


258 common dates
GSW   sd 4.307 bp
DGS10 sd 4.094 bp
correlation of daily changes: 0.985


In [ ]:
# DV01 calculation 

def mod_duration_par(y, T=10.0):
    """Modified duration of a semiannual par coupon bond. y decimal, T years"""
    D_mod = (1 - (1 + y/2)**(-2*T)) / y
    return D_mod

y_yield = float(y10w.mean()) / 100    
print(f"average 10y yield: {y_yield*100:.2f}%")

print(mod_duration_par(0.00001, 10))  # expect ≈ 10.0  (no coupon → duration = maturity)

average 10y yield: 4.38%
9.99947501931775


In [ ]:
# Calculating the gap scale, g

maturity_years = 9.75 #first off-the-run: ~1-3 months past issue
LAMBDA = 0.1 #hard coded

mod_duration = mod_duration_par(y_yield, maturity_years)
points_per_bp = mod_duration * 0.0001 * 100
sigma = sigma_bp * points_per_bp
g = LAMBDA * sigma

print(f"mod duration  : {mod_duration:.3f}  ({maturity_years}y at {y_yield*100:.2f}%)")
print(f"points per bp : {points_per_bp:.4f}  (DV01 ${points_per_bp*10_000:,.0f} per $1mm)")
print(f"sigma         : {sigma:.4f} pts/day ({sigma*32:.2f}/32nds)")
print(f"g             : {g:.4f} pts        ({g*32:.2f}/32nds)")
print(f"g on $10mm    : ${g/100*10_000_000:,.0f}")
print(f"ticks per g   : {g/0.001:.1f}")

assert 0.3  <= sigma <= 0.8,  f"GATE FAIL: sigma={sigma:.4f} outside 0.3-0.8"
assert 0.02 <= g     <= 0.10, f"GATE FAIL: g={g:.4f} outside 0.02-0.10"
print("\ngates pass")

mod duration  : 7.865  (9.75y at 4.38%)
points per bp : 0.0786  (DV01 $786 per $1mm)
sigma         : 0.3387 pts/day (10.84/32nds)
g             : 0.0339 pts        (1.08/32nds)
g on $10mm    : $3,387
ticks per g   : 33.9

gates pass


In [ ]:
# Check against config.yaml (written on 2026-09-22; this cell no longer overwrites it)
import yaml

cal = yaml.safe_load(open("config.yaml"))["calibration"]
rows = [("window end", str(y10w.index.max().date()), cal["window_end"]),
        ("observations", len(y10w), cal["n_obs"]),
        ("sigma_bp", round(float(sigma_bp), 4), cal["sigma_bp"]),
        ("mean yield %", round(float(y_yield) * 100, 3), cal["duration_at_yield_pct"]),
        ("mod duration", round(float(mod_duration), 4), cal["mod_duration"]),
        ("g (points)", round(float(g), 7), cal["g_points"])]
print(f"{'':<14}{'recomputed':>14}{'config.yaml':>14}")
for name, now, saved in rows:
    print(f"{name:<14}{str(now):>14}{str(saved):>14}")

                  recomputed   config.yaml
window end        2026-09-11    2026-09-11
observations             258           258
sigma_bp              4.3068        4.3068
mean yield %           4.384         4.384
mod duration           7.865         7.865
g (points)         0.0338731     0.0338731
